# 第 04 章：高级工具调用 (Smart Tooling)

根据讲义，本章你将掌握以下核心能力：
- **Pydantic v2 契约建模**：建立坚不可摧的参数校验层。
- **运行时上下文注入**：利用 `InjectedState` 实现安全权限透传。
- **防御式编程实操**：应对小模型的参数生成幻觉。
- **揭秘调度者**：亲手实现一个“零魔法”的工具自动调度器。

## 1. 统一模型声明
配置环境变量，并加载具备工具调用能力的 DeepSeek 推理基座。

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(
    model="deepseek-chat", 
    api_key=os.getenv("DEEPSEEK_API_KEY"), 
    base_url="https://api.deepseek.com"
)
print("模型实例连接就绪。")

## 2. 契约化建模对比：docstring vs Pydantic
通过 Pydantic 显式定义 `args_schema`，让 LLM 面对复杂约束（如数值范围）时表现更稳定。

In [ ]:
from langchain_core.tools import tool
from pydantic import BaseModel, Field
import json

class SearchArgs(BaseModel):
    query: str = Field(description="搜索关键词")
    limit: int = Field(default=5, description="结果数量", ge=1, le=10)

@tool(args_schema=SearchArgs)
def smart_search(query: str, limit: int = 5):
    """
    增强型搜索工具，具有极致的参数校验稳定性。
    """
    return f"[执行搜索] 关键词: {query}, 数量上限: {limit}"

print("--- LLM 接收到的 JSON Schema (给 LLM 看的合同) --- ")
print(json.dumps(smart_search.args_schema.model_json_schema(), indent=2, ensure_ascii=False))

## 3. 安全穿透：使用 InjectedState 注入上下文
我们先定义一个需要 `user_id` 的敏感工具。

In [ ]:
from typing import Annotated
from langgraph.prebuilt import InjectedState

@tool
def secure_delete(item_id: str, user_id: Annotated[str, InjectedState("user_id")]):
    """
    安全删除指定项目。LLM 严禁生成 user_id，该字段由系统自动注入。
    """
    return f"[审计成功] 用户 {user_id} 已发起对项目 {item_id} 的删除操作。"

from langchain_core.utils.function_calling import convert_to_openai_tool
llm_spec = convert_to_openai_tool(secure_delete)
print(f"LLM 真正能看到的参数: {list(llm_spec['function']['parameters']['properties'].keys())}")
print(f"开发环境实际需要的参数: {list(secure_delete.args_schema.model_json_schema()['properties'].keys())}")

## 4. 重点：揭秘调度者 (The Dispatcher)
如果没有 `create_agent` 的黑盒，我们如何手动让 LLM 的**意图**转化为**代码执行**？

仔细观察下方的 `manual_dispatch` 逻辑，它就是那只“隐形的手”。

In [ ]:
from langchain_core.messages import AIMessage, ToolMessage

def manual_dispatch(ai_msg: AIMessage, state: dict):
    """
    模拟运行时调度器：读取意图，注入状态，真正执行函数。
    """
    results = []
    for tool_call in ai_msg.tool_calls:
        # 1. 拦截与路由：目前只模拟这一个函数
        if tool_call["name"] in ["secure_delete", "file_operator"]:
            final_args = tool_call["args"].copy()
            
            # 2. 注入隐私状态（核心魔法发生点）
            final_args["user_id"] = state["current_user_id"]
            
            # 3. 动态寻找函数名并点火执行
            fn_map = {"secure_delete": secure_delete, "file_operator": file_operator}
            res = fn_map[tool_call["name"]].invoke(final_args)
            results.append(ToolMessage(content=res, tool_call_id=tool_call["id"]))
            
    return results

print("手动调度器 (Dispatcher) 已就绪。")

## 5. 本章 Lab：智能文件权限管理系统 (实战)

我们将构建一个模拟的文件操作工具，它的权限校验完全依赖于系统注入的上下文。

In [ ]:
# 1. 模拟一个权限数据库：只有 admin 能读写 secret.txt
ACL = {
    "admin_9527": ["readme.md", "secret.txt"],
    "guest_001": ["readme.md"]
}

# 2. 定义具备审计能力的工具
@tool
def file_operator(file_name: str, action: str, user_id: Annotated[str, InjectedState("user_id")]):
    """
    文件操作器。支持 action 为 'read' 或 'delete'。
    """
    print(f"[系统审计] 开始校验用户 {user_id} 对 {file_name} 的 {action} 权限...")
    
    allowed_files = ACL.get(user_id, [])
    if file_name not in allowed_files:
        return f"错误：用户 {user_id} 权限不足，无法对 {file_name} 执行 {action} 操作。"
    
    return f"成功：用户 {user_id} 已完成对 {file_name} 的 {action} 操作。"

# 3. 模拟测试：假设模型想读 secret.txt
mock_ai_intent = AIMessage(
    content="",
    tool_calls=[{"name": "file_operator", "args": {"file_name": "secret.txt", "action": "read"}, "id": "lab_1"}]
)

print("--- 场景 A：当系统发现用户是客 (guest_001) 时 ---")
res_guest = manual_dispatch(mock_ai_intent, {"current_user_id": "guest_001"})
print(res_guest[0].content)

print("\n--- 场景 B：当系统确认用户是管理员 (admin_9527) 时 ---")
res_admin = manual_dispatch(mock_ai_intent, {"current_user_id": "admin_9527"})
print(res_admin[0].content)